# <center> Fruit Quality and Adulteration Classifer </center>
---- 

### 1. Imports

In [9]:
import os
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder

### 2. Data Transformation Pipeline

In [10]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

### 2.1 Custom Dataset (Fruit -> Disease Category)

Dataset ka structure aisa hai:

```
Dataset/train/Apple/Fresh/...
Dataset/train/Apple/Rotten/...
Dataset/train/Apple/Formalin-mixed/...
Dataset/train/Banana/Fresh/...
... (waghera)
```

`ImageFolder` sirf top-level folders (fruit names) ko class maan raha tha, isi wajah se 5 classes ban rahi thi (Apple, Banana, Grape, Mango, Orange) jabke hum ne sirf 3 disease categories predict karni hain: **Formalin-mixed, Fresh, Rotten**.

Is problem ko fix karne ke liye neeche ek custom `Dataset` class likhi hai jo har fruit folder ke andar jaake disease-subfolder (Formalin-mixed / Fresh / Rotten) ke naam se label assign karti hai, fruit type ko ignore karte hue.

In [11]:
class FruitDiseaseDataset(Dataset):
    """
    Recursively walks: root_dir/<fruit>/<disease_category>/<image>
    and returns label based on disease_category (Formalin-mixed, Fresh, Rotten),
    ignoring the fruit type.
    """

    classes = ['Fresh', 'Rotten']

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.samples = []

        for fruit_folder in sorted(os.listdir(root_dir)):
            fruit_path = os.path.join(root_dir, fruit_folder)
            if not os.path.isdir(fruit_path):
                continue

            for disease_folder in sorted(os.listdir(fruit_path)):
                if disease_folder not in self.class_to_idx:
                    continue

                disease_path = os.path.join(fruit_path, disease_folder)
                if not os.path.isdir(disease_path):
                    continue

                label = self.class_to_idx[disease_folder]

                for fname in os.listdir(disease_path):
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.samples.append((os.path.join(disease_path, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

### 3. Load Train, Test and Validation Data

In [12]:
train_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\Fruits-Diseases-Classification-CNN\Dataset\train"
test_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\Fruits-Diseases-Classification-CNN\Dataset\test"
val_dataset_path = r"C:\Users\TAQICOMPUTERS\Desktop\Fruits-Diseases-Classification-CNN\Dataset\valid"

train_data = FruitDiseaseDataset(train_dataset_path, transform=transform)
test_data = FruitDiseaseDataset(test_dataset_path, transform=transform)
val_data = FruitDiseaseDataset(val_dataset_path, transform=transform)

print(train_data.classes)
print(train_data.class_to_idx)
print(f"Total training images: {len(train_data)}")

['Fresh', 'Rotten']
{'Fresh': 0, 'Rotten': 1}
Total training images: 4884


### 4. Pytorch Data Loader

In [13]:
train_loader = DataLoader(train_data, batch_size=95, shuffle=True)
test_loader = DataLoader(test_data, batch_size=95, shuffle=False)
val_loader = DataLoader(val_data, batch_size=95, shuffle=False)

### 5. Model Architecture

In [14]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(16*16*128, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x


### 6. Build Model

In [15]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 7. Training Loop

In [16]:
epochs = 10
model.train()

print("------- Training Started -------")
for epoch in range(epochs):
    epoch_train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()
    
    avg_epoch_loss = epoch_train_loss / len(train_loader)
    print(f"Epoch: {epoch + 1} \ {epochs} Loss: {avg_epoch_loss:.4f}")

------- Training Started -------
Epoch: 1 \ 10 Loss: 0.5858
Epoch: 2 \ 10 Loss: 0.2651
Epoch: 3 \ 10 Loss: 0.1998
Epoch: 4 \ 10 Loss: 0.1576
Epoch: 5 \ 10 Loss: 0.1602
Epoch: 6 \ 10 Loss: 0.1348
Epoch: 7 \ 10 Loss: 0.1076
Epoch: 8 \ 10 Loss: 0.1063
Epoch: 9 \ 10 Loss: 0.0880
Epoch: 10 \ 10 Loss: 0.0770


### 8. Save Model Paramters

In [18]:
model.eval() 
correct_labels = 0
total_labels = 0

with torch.no_grad(): 
    for images, labels in val_loader:
        outputs = model(images)                
        _, predicted = torch.max(outputs, 1)   
        correct_labels += (predicted == labels).sum().item()  
        total_labels += labels.size(0)                        

accuracy = (correct_labels / total_labels) * 100
print(f"\nValidation Accuracy: {accuracy:.2f}%")

torch.save(model.state_dict(), 'fruit_quality_and_adulteration_classifier.pth')
print("Model saved successfully as 'fruit_quality_and_adulteration_classifier.pth'!")


Validation Accuracy: 93.80%
Model saved successfully as 'fruit_quality_and_adulteration_classifier.pth'!
